In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
import mlflow
import os
from dotenv import load_dotenv;load_dotenv()

True

In [11]:
df = pd.read_excel("../data/delivery_data_24features.xlsx")

In [12]:
# Encode categorical columns
categorical_cols = df.select_dtypes(include=['object']).columns
df_encoded = df.copy()
for col in categorical_cols:
    df_encoded[col] = LabelEncoder().fit_transform(df_encoded[col].astype(str))

# Split data
X = df_encoded.drop('Time_taken(min)', axis=1)
y = df_encoded['Time_taken(min)']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [13]:
mlflow.set_tracking_uri("https://dagshub.com/iamprashantjain/Swiggy-Delivery-Time-Prediction.mlflow")
os.environ["MLFLOW_TRACKING_USERNAME"] = os.getenv("MLFLOW_TRACKING_USERNAME", "iamprashantjain")
os.environ["MLFLOW_TRACKING_PASSWORD"] = os.getenv("MLFLOW_TRACKING_PASSWORD", "")

mlflow.set_experiment("base_model")

<Experiment: artifact_location='mlflow-artifacts:/e62f27fb9c1d473b8a6c9af4abcfdf63', creation_time=1777457202261, experiment_id='3', last_update_time=1777457202261, lifecycle_stage='active', name='base_model', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [14]:
with mlflow.start_run():    
    # Train model
    model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    
    # Predict
    y_pred = model.predict(X_test)
    
    # Calculate metrics
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    # Log to MLflow
    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("n_features", X.shape[1])
    mlflow.log_metric("MAE", mae)
    mlflow.log_metric("RMSE", rmse)
    mlflow.log_metric("R2", r2)
    
    # Log model
    mlflow.sklearn.log_model(model, "random_forest_model")
    
    print(f"\n{'='*50}")
    print(f"BASELINE RESULTS - ALL 24 FEATURES")
    print(f"{'='*50}")
    print(f"   MAE:  {mae:.2f} minutes")
    print(f"   RMSE: {rmse:.2f} minutes")
    print(f"   R²:   {r2:.4f}")
    print(f"\n Model logged to MLflow")
    print(f"{'='*50}")

d:\Swiggy-Delivery-Time-Prediction\venv\lib\site-packages\_distutils_hack\__init__.py:18: UserWarning: Distutils was imported before Setuptools, but importing Setuptools also replaces the `distutils` module in `sys.modules`. This may lead to undesirable behaviors or errors. To avoid these issues, avoid using distutils directly, ensure that setuptools is installed in the traditional way (e.g. not an editable install), and/or make sure that setuptools is always imported before distutils.
  warnings.warn(
d:\Swiggy-Delivery-Time-Prediction\venv\lib\site-packages\_distutils_hack\__init__.py:33: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")
2026/04/29 15:55:40 WARNING mlflow.models.model: Input example should be provided to infer model signature if the model signature is not provided when logging the model.



BASELINE RESULTS - ALL 24 FEATURES
   MAE:  3.45 minutes
   RMSE: 4.33 minutes
   R²:   0.7852

✅ Model logged to MLflow


2026/04/29 15:55:41 INFO mlflow.tracking._tracking_service.client: 🏃 View run big-hen-28 at: https://dagshub.com/iamprashantjain/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/3/runs/ac3c3128bd864181a20a51fe0052ed4a.
2026/04/29 15:55:41 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: https://dagshub.com/iamprashantjain/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/3.


### With more important features

In [15]:
final_features = [
    # TIER 1 - Essential (10 features)
    'Delivery_person_Ratings',
    'multiple_deliveries', 
    'Weatherconditions',
    'Road_traffic_density',
    'Delivery_person_Age',
    'Vehicle_condition',
    'order_hour',
    'order_day',
    'time_category',
    'Festival',
    
    # TIER 2 - Hidden Gems (4 features)
    'Type_of_vehicle',
    'order_dayofweek',
    'is_rush_hour',
    'order_day_name'
]

target = 'Time_taken(min)'

In [16]:
# Why These 14?
# Reason	Evidence
# Tier 1	High Mutual Info scores + statistically significant (passed shadow test)
# Tier 2	Low linear correlation but high predictive power (your "hidden gems")
# RFECV confirmed	Optimal number was 14 features (from your output)

In [17]:
exclude_features = [
    'is_peak_hour',        # Redundant with is_rush_hour?
    'is_late_night',       # Low importance
    'season',              # Very low MI (0.002)
    'order_month',         # Very low MI (0.002)
    'preparation_time_minutes',  # Extremely low MI (0.0003)
    'delivery_city',       # Not in top tiers
    'order_minute',        # Not in top tiers  
    'Type_of_order',       # Not in top tiers
    'order_year',          # Likely constant/low variance
    'is_weekend'           # MI was 0.000
]

In [18]:
# Create final dataset
df_final_model = df[final_features + [target]].copy()

# Save for modeling
df_final_model.to_excel("../data/final_features_14.xlsx", index=False)

In [19]:
# Encode categorical columns
categorical_cols = df_final_model.select_dtypes(include=['object']).columns
df_final_model_encoded = df_final_model.copy()
for col in categorical_cols:
    df_final_model_encoded[col] = LabelEncoder().fit_transform(df_final_model_encoded[col].astype(str))

# Split data
X = df_final_model_encoded.drop('Time_taken(min)', axis=1)
y = df_final_model_encoded['Time_taken(min)']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

mlflow.set_tracking_uri("https://dagshub.com/iamprashantjain/Swiggy-Delivery-Time-Prediction.mlflow")
os.environ["MLFLOW_TRACKING_USERNAME"] = os.getenv("MLFLOW_TRACKING_USERNAME", "iamprashantjain")
os.environ["MLFLOW_TRACKING_PASSWORD"] = os.getenv("MLFLOW_TRACKING_PASSWORD", "")

mlflow.set_experiment("base_model_with_14_important_features")

with mlflow.start_run():
    # Train model
    model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    
    # Predict
    y_pred = model.predict(X_test)
    
    # Calculate metrics
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    # Log to MLflow
    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("n_features", X.shape[1])
    mlflow.log_metric("MAE", mae)
    mlflow.log_metric("RMSE", rmse)
    mlflow.log_metric("R2", r2)
    
    # Log model
    mlflow.sklearn.log_model(model, "random_forest_model")
    
    print(f"\n{'='*50}")
    print(f"BASELINE RESULTS - ALL 24 FEATURES")
    print(f"{'='*50}")
    print(f"   MAE:  {mae:.2f} minutes")
    print(f"   RMSE: {rmse:.2f} minutes")
    print(f"   R²:   {r2:.4f}")
    print(f"\n Model logged to MLflow")
    print(f"{'='*50}")

2026/04/29 16:01:13 INFO mlflow.tracking.fluent: Experiment with name 'base_model_with_14_important_features' does not exist. Creating a new experiment.
d:\Swiggy-Delivery-Time-Prediction\venv\lib\site-packages\_distutils_hack\__init__.py:18: UserWarning: Distutils was imported before Setuptools, but importing Setuptools also replaces the `distutils` module in `sys.modules`. This may lead to undesirable behaviors or errors. To avoid these issues, avoid using distutils directly, ensure that setuptools is installed in the traditional way (e.g. not an editable install), and/or make sure that setuptools is always imported before distutils.
  warnings.warn(
d:\Swiggy-Delivery-Time-Prediction\venv\lib\site-packages\_distutils_hack\__init__.py:33: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")
2026/04/29 16:05:26 WARNING mlflow.models.model: Input example should be provided to infer model signature if the model signature is not provided w


BASELINE RESULTS - ALL 24 FEATURES
   MAE:  3.36 minutes
   RMSE: 4.23 minutes
   R²:   0.7950

 Model logged to MLflow


2026/04/29 16:05:27 INFO mlflow.tracking._tracking_service.client: 🏃 View run spiffy-mare-654 at: https://dagshub.com/iamprashantjain/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/4/runs/0294971bf0264ade86bfb13a0205aa2a.
2026/04/29 16:05:27 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: https://dagshub.com/iamprashantjain/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/4.


### best algorithms

In [20]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import lightgbm as lgb
import mlflow
import os
import numpy as np
import pandas as pd

# Define algorithms dictionary
algorithms = {
    'RandomForest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'GradientBoosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'XGBoost': xgb.XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1, verbosity=0),
    'LightGBM': lgb.LGBMRegressor(n_estimators=100, random_state=42, n_jobs=-1, verbose=-1),
    'AdaBoost': AdaBoostRegressor(n_estimators=100, random_state=42),
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=1.0, random_state=42),
    'ElasticNet': ElasticNet(alpha=1.0, l1_ratio=0.5, random_state=42),
    'DecisionTree': DecisionTreeRegressor(random_state=42),
    'KNN': KNeighborsRegressor(n_neighbors=5),
    'SVR': SVR()
}

# Store results for comparison
results = []

# Start MLflow run
mlflow.set_experiment("multiple_algorithm_14_features")

with mlflow.start_run(run_name="All Models Comparison - 14 Features") as parent_run:
    
    for algo_name, model in algorithms.items():
        with mlflow.start_run(run_name=algo_name, nested=True) as child_run:
            
            print(f"\n{'='*60}")
            print(f"📊 Training: {algo_name}")
            print(f"{'='*60}")
            
            try:
                # Train model
                model.fit(X_train, y_train)
                
                # Predict
                y_pred = model.predict(X_test)
                
                # Calculate metrics
                mae = mean_absolute_error(y_test, y_pred)
                rmse = np.sqrt(mean_squared_error(y_test, y_pred))
                r2 = r2_score(y_test, y_pred)
                
                # Calculate MAPE (Mean Absolute Percentage Error)
                mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
                
                # Store results
                results.append({
                    'Model': algo_name,
                    'MAE (min)': mae,
                    'RMSE (min)': rmse,
                    'R² Score': r2,
                    'MAPE (%)': mape
                })
                
                # Log parameters
                mlflow.log_param("model_type", algo_name)
                mlflow.log_param("n_features", X_train.shape[1])
                
                # Log algorithm-specific parameters
                params_to_log = {}
                if hasattr(model, 'get_params'):
                    params = model.get_params()
                    if algo_name == 'RandomForest' or algo_name == 'GradientBoosting' or algo_name == 'AdaBoost':
                        params_to_log = {k: v for k, v in params.items() if k in ['n_estimators', 'learning_rate', 'max_depth']}
                    elif algo_name == 'XGBoost' or algo_name == 'LightGBM':
                        params_to_log = {k: v for k, v in params.items() if k in ['n_estimators', 'learning_rate', 'max_depth']}
                    elif algo_name in ['Ridge', 'Lasso', 'ElasticNet']:
                        params_to_log = {k: v for k, v in params.items() if k in ['alpha', 'l1_ratio']}
                    elif algo_name == 'KNN':
                        params_to_log = {'n_neighbors': model.n_neighbors}
                
                for param_name, param_value in params_to_log.items():
                    mlflow.log_param(param_name, param_value)
                
                # Log metrics
                mlflow.log_metric("MAE", mae)
                mlflow.log_metric("RMSE", rmse)
                mlflow.log_metric("R2", r2)
                mlflow.log_metric("MAPE", mape)
                
                # Log model
                mlflow.sklearn.log_model(model, f"{algo_name}_model")
                
                # Print results
                print(f"   ✅ MAE:  {mae:.2f} minutes")
                print(f"   ✅ RMSE: {rmse:.2f} minutes")
                print(f"   ✅ R²:   {r2:.4f}")
                print(f"   ✅ MAPE: {mape:.2f}%")
                print(f"   📝 Logged to MLflow")
                
            except Exception as e:
                print(f"   ❌ Error: {str(e)[:100]}")
                results.append({
                    'Model': algo_name,
                    'MAE (min)': np.nan,
                    'RMSE (min)': np.nan,
                    'R² Score': np.nan,
                    'MAPE (%)': np.nan
                })

# ============================================
# RESULTS COMPARISON
# ============================================
print(f"\n{'='*80}")
print("🏆 ALGORITHM COMPARISON RESULTS (14 Features)")
print(f"{'='*80}")

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('R² Score', ascending=False)

print("\n📊 Sorted by R² Score (Best to Worst):")
print(results_df.to_string(index=False))

# Find and display best model
best_model_row = results_df.iloc[0]
print(f"\n{'='*80}")
print(f"🎯 BEST MODEL: {best_model_row['Model']}")
print(f"{'='*80}")
print(f"   MAE:  {best_model_row['MAE (min)']:.2f} minutes")
print(f"   RMSE: {best_model_row['RMSE (min)']:.2f} minutes")
print(f"   R²:   {best_model_row['R² Score']:.4f}")
print(f"   MAPE: {best_model_row['MAPE (%)']:.2f}%")

# Optional: Visual comparison
print(f"\n📈 QUICK INSIGHTS:")
print("-" * 40)

# Group by model family
tree_models = results_df[results_df['Model'].isin(['RandomForest', 'GradientBoosting', 'XGBoost', 'LightGBM', 'AdaBoost', 'DecisionTree'])]
linear_models = results_df[results_df['Model'].isin(['LinearRegression', 'Ridge', 'Lasso', 'ElasticNet'])]
other_models = results_df[results_df['Model'].isin(['KNN', 'SVR'])]

if len(tree_models) > 0:
    print(f"🌲 Best Tree-based: {tree_models.iloc[0]['Model']} (R²={tree_models.iloc[0]['R² Score']:.4f})")
if len(linear_models) > 0:
    print(f"📐 Best Linear: {linear_models.iloc[0]['Model']} (R²={linear_models.iloc[0]['R² Score']:.4f})")
if len(other_models) > 0:
    print(f"🔧 Best Other: {other_models.iloc[0]['Model']} (R²={other_models.iloc[0]['R² Score']:.4f})")

print(f"\n✅ All results logged to MLflow at: {mlflow.get_tracking_uri()}")

2026/04/29 16:08:48 INFO mlflow.tracking.fluent: Experiment with name 'multiple_algorithm_14_features' does not exist. Creating a new experiment.



📊 Training: RandomForest


d:\Swiggy-Delivery-Time-Prediction\venv\lib\site-packages\_distutils_hack\__init__.py:18: UserWarning: Distutils was imported before Setuptools, but importing Setuptools also replaces the `distutils` module in `sys.modules`. This may lead to undesirable behaviors or errors. To avoid these issues, avoid using distutils directly, ensure that setuptools is installed in the traditional way (e.g. not an editable install), and/or make sure that setuptools is always imported before distutils.
  warnings.warn(
d:\Swiggy-Delivery-Time-Prediction\venv\lib\site-packages\_distutils_hack\__init__.py:33: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")
2026/04/29 16:14:46 WARNING mlflow.models.model: Input example should be provided to infer model signature if the model signature is not provided when logging the model.


   ✅ MAE:  3.36 minutes
   ✅ RMSE: 4.23 minutes
   ✅ R²:   0.7950
   ✅ MAPE: 14.47%
   📝 Logged to MLflow


2026/04/29 16:14:46 INFO mlflow.tracking._tracking_service.client: 🏃 View run RandomForest at: https://dagshub.com/iamprashantjain/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/5/runs/2ba4241a176d4704b849e1c21ecc4dc9.
2026/04/29 16:14:46 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: https://dagshub.com/iamprashantjain/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/5.


...


📊 Training: SVR


d:\Swiggy-Delivery-Time-Prediction\venv\lib\site-packages\_distutils_hack\__init__.py:18: UserWarning: Distutils was imported before Setuptools, but importing Setuptools also replaces the `distutils` module in `sys.modules`. This may lead to undesirable behaviors or errors. To avoid these issues, avoid using distutils directly, ensure that setuptools is installed in the traditional way (e.g. not an editable install), and/or make sure that setuptools is always imported before distutils.
  warnings.warn(
d:\Swiggy-Delivery-Time-Prediction\venv\lib\site-packages\_distutils_hack\__init__.py:33: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")
2026/04/29 16:24:04 WARNING mlflow.models.model: Input example should be provided to infer model signature if the model signature is not provided when logging the model.


   ✅ MAE:  5.62 minutes
   ✅ RMSE: 7.03 minutes
   ✅ R²:   0.4344
   ✅ MAPE: 24.51%
   📝 Logged to MLflow


2026/04/29 16:24:04 INFO mlflow.tracking._tracking_service.client: 🏃 View run SVR at: https://dagshub.com/iamprashantjain/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/5/runs/0ad74120382842ee95754402d3d83d45.
2026/04/29 16:24:04 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: https://dagshub.com/iamprashantjain/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/5.
2026/04/29 16:24:05 INFO mlflow.tracking._tracking_service.client: 🏃 View run All Models Comparison - 14 Features at: https://dagshub.com/iamprashantjain/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/5/runs/6c2140a4544349159ffe9d51ef741169.
2026/04/29 16:24:05 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: https://dagshub.com/iamprashantjain/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/5.



🏆 ALGORITHM COMPARISON RESULTS (14 Features)

📊 Sorted by R² Score (Best to Worst):
           Model  MAE (min)  RMSE (min)  R² Score  MAPE (%)
         XGBoost   3.256473    4.045761  0.812840 14.125765
        LightGBM   3.283728    4.071867  0.810417 14.358544
    RandomForest   3.356661    4.233786  0.795040 14.474066
GradientBoosting   3.963308    4.969172  0.717655 17.499551
    DecisionTree   4.296904    5.670834  0.632290 18.356334
             KNN   4.839538    6.181535  0.563078 21.015898
        AdaBoost   5.214736    6.329814  0.541865 24.273543
LinearRegression   5.348835    6.683601  0.489222 23.784855
           Ridge   5.348888    6.683619  0.489219 23.785382
             SVR   5.616971    7.033059  0.434412 24.506046
      ElasticNet   6.328942    7.802963  0.303806 28.215956
           Lasso   6.349347    7.820265  0.300715 28.269264

🎯 BEST MODEL: XGBoost
   MAE:  3.26 minutes
   RMSE: 4.05 minutes
   R²:   0.8128
   MAPE: 14.13%

📈 QUICK INSIGHTS:
-----------------

### hyper parameter tuning on xgboost

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import mlflow
import os

mlflow.set_experiment("XGBoost_Hyperparameter_Tuning")

# Define parameter grid for XGBoost
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7, 9],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'min_child_weight': [1, 3, 5]
}

# Start parent run
with mlflow.start_run(run_name="XGBoost_Hyperparameter_Tuning") as parent_run:
    
    # Perform grid search with cross-validation
    xgb_model = xgb.XGBRegressor(random_state=42, n_jobs=-1, verbosity=0)
    grid_search = GridSearchCV(
        estimator=xgb_model,
        param_grid=param_grid,
        cv=5,
        scoring='r2',
        n_jobs=-1,
        verbose=2
    )
    
    print("\n📊 Running GridSearchCV...")
    grid_search.fit(X_train, y_train)
    
    # Log each parameter combination as child runs
    print("\n📝 Logging individual runs to MLflow...")
    for i, (params, mean_score, std_score) in enumerate(zip(
        grid_search.cv_results_['params'],
        grid_search.cv_results_['mean_test_score'],
        grid_search.cv_results_['std_test_score']
    )):
        with mlflow.start_run(run_name=f"XGB_{params}", nested=True):
            # Train model with these parameters
            model = xgb.XGBRegressor(**params, random_state=42, n_jobs=-1, verbosity=0)
            model.fit(X_train, y_train)
            
            # Evaluate on test set
            y_pred = model.predict(X_test)
            mae = mean_absolute_error(y_test, y_pred)
            rmse = np.sqrt(mean_squared_error(y_test, y_pred))
            r2 = r2_score(y_test, y_pred)
            mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
            
            # Log parameters and metrics
            mlflow.log_params(params)
            mlflow.log_metric("mean_cv_r2", mean_score)
            mlflow.log_metric("std_cv_r2", std_score)
            mlflow.log_metric("test_MAE", mae)
            mlflow.log_metric("test_RMSE", rmse)
            mlflow.log_metric("test_R2", r2)
            mlflow.log_metric("test_MAPE", mape)
            
            if i % 50 == 0:
                print(f"   Logged {i+1}/{len(grid_search.cv_results_['params'])} combinations...")
    
    # Get best parameters
    best_params = grid_search.best_params_
    best_score = grid_search.best_score_
    
    # Log best parameters and score to parent run
    mlflow.log_params(best_params)
    mlflow.log_metric("best_cv_r2_score", best_score)
    
    # Evaluate best model on test set
    best_model = grid_search.best_estimator_
    y_pred_best = best_model.predict(X_test)
    
    best_mae = mean_absolute_error(y_test, y_pred_best)
    best_rmse = np.sqrt(mean_squared_error(y_test, y_pred_best))
    best_r2 = r2_score(y_test, y_pred_best)
    best_mape = np.mean(np.abs((y_test - y_pred_best) / y_test)) * 100
    
    # Log best model test metrics
    mlflow.log_metric("best_test_MAE", best_mae)
    mlflow.log_metric("best_test_RMSE", best_rmse)
    mlflow.log_metric("best_test_R2", best_r2)
    mlflow.log_metric("best_test_MAPE", best_mape)
    
    # Log feature importance of best model
    feature_importance = pd.DataFrame({
        'feature': features,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    # Save and log feature importance as artifact
    feature_importance.to_csv('xgboost_feature_importance.csv', index=False)
    mlflow.log_artifact('xgboost_feature_importance.csv')
    
    # Save and log best model
    mlflow.sklearn.log_model(best_model, "best_xgboost_model")
    
    # Print results
    print("\n" + "="*60)
    print("🎯 HYPERPARAMETER TUNING COMPLETE!")
    print("="*60)
    print(f"\n✅ Best Parameters found:")
    for param, value in best_params.items():
        print(f"   • {param}: {value}")
    
    print(f"\n📊 Best Cross-Validation R² Score: {best_score:.4f}")
    
    print(f"\n🚀 Best Model Performance on Test Set:")
    print(f"   • MAE:  {best_mae:.2f} minutes")
    print(f"   • RMSE: {best_rmse:.2f} minutes")
    print(f"   • R²:   {best_r2:.4f}")
    print(f"   • MAPE: {best_mape:.2f}%")
    
    print(f"\n📈 Feature Importance (Top 10):")
    print(feature_importance.head(10).to_string(index=False))
    
    # Comparison with default XGBoost
    print("\n" + "="*60)
    print("📊 COMPARISON: Default vs Tuned XGBoost")
    print("="*60)
    
    # Train default model for comparison
    default_model = xgb.XGBRegressor(random_state=42, n_jobs=-1, verbosity=0)
    default_model.fit(X_train, y_train)
    y_pred_default = default_model.predict(X_test)
    
    default_mae = mean_absolute_error(y_test, y_pred_default)
    default_r2 = r2_score(y_test, y_pred_default)
    
    print(f"\nDefault XGBoost:")
    print(f"   MAE: {default_mae:.2f} minutes")
    print(f"   R²:  {default_r2:.4f}")
    
    print(f"\nTuned XGBoost:")
    print(f"   MAE: {best_mae:.2f} minutes")
    print(f"   R²:  {best_r2:.4f}")
    
    print(f"\n📈 Improvement:")
    print(f"   MAE: {(default_mae - best_mae):.2f} minutes better")
    print(f"   R²:  {(best_r2 - default_r2):.4f} points higher")
    
    print("\n" + "="*60)
    print(f"🔗 View results at: {mlflow.get_tracking_uri()}")
    print("="*60)

2026/04/29 16:32:48 INFO mlflow.tracking.fluent: Experiment with name 'XGBoost_Hyperparameter_Tuning' does not exist. Creating a new experiment.



📊 Running GridSearchCV...
Fitting 5 folds for each of 1296 candidates, totalling 6480 fits

📝 Logging individual runs to MLflow...
   Logged 1/1296 combinations...


2026/04/29 17:47:21 INFO mlflow.tracking._tracking_service.client: 🏃 View run XGB_{'colsample_bytree': 0.6, 'learning_rate': 0.01, 'max_depth': 3, 'min_child_weight': 1, 'n_estimators': 100, 'subsample': 0.6} at: https://dagshub.com/iamprashantjain/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/6/runs/0d93b2a0e47d4e3b96fcbbfa67e163da.
2026/04/29 17:47:21 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: https://dagshub.com/iamprashantjain/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/6.
2026/04/29 17:47:26 INFO mlflow.tracking._tracking_service.client: 🏃 View run XGB_{'colsample_bytree': 0.6, 'learning_rate': 0.01, 'max_depth': 3, 'min_child_weight': 1, 'n_estimators': 100, 'subsample': 0.8} at: https://dagshub.com/iamprashantjain/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/6/runs/bcfa44e984ba47c3ab4cb394c5d4aa44.
2026/04/29 17:47:26 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: https://dagshub.com/iamprashantjain/Swiggy-D

   Logged 51/1296 combinations...


2026/04/29 17:55:36 INFO mlflow.tracking._tracking_service.client: 🏃 View run XGB_{'colsample_bytree': 0.6, 'learning_rate': 0.01, 'max_depth': 5, 'min_child_weight': 5, 'n_estimators': 200, 'subsample': 1.0} at: https://dagshub.com/iamprashantjain/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/6/runs/b4c42d8dc9ba4783888270d03e0259cf.
2026/04/29 17:55:36 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: https://dagshub.com/iamprashantjain/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/6.
2026/04/29 17:55:46 INFO mlflow.tracking._tracking_service.client: 🏃 View run XGB_{'colsample_bytree': 0.6, 'learning_rate': 0.01, 'max_depth': 5, 'min_child_weight': 5, 'n_estimators': 300, 'subsample': 0.6} at: https://dagshub.com/iamprashantjain/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/6/runs/0dbc88c11e1a42a5ad8856b8da51c417.
2026/04/29 17:55:46 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: https://dagshub.com/iamprashantjain/Swiggy-D

...

2026/04/29 21:01:11 INFO mlflow.tracking._tracking_service.client: 🏃 View run XGB_{'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 7, 'min_child_weight': 3, 'n_estimators': 300, 'subsample': 0.8} at: https://dagshub.com/iamprashantjain/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/6/runs/c7e44b2bc2334c51a37ef46e02e9d9bb.
2026/04/29 21:01:11 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: https://dagshub.com/iamprashantjain/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/6.
2026/04/29 21:01:21 INFO mlflow.tracking._tracking_service.client: 🏃 View run XGB_{'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 7, 'min_child_weight': 3, 'n_estimators': 300, 'subsample': 1.0} at: https://dagshub.com/iamprashantjain/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/6/runs/7d80f250010a442b95f2fab5d0844bdf.
2026/04/29 21:01:21 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: https://dagshub.com/iamprashantjain/Swiggy-Del

   Logged 1201/1296 combinations...


2026/04/29 21:09:33 INFO mlflow.tracking._tracking_service.client: 🏃 View run XGB_{'colsample_bytree': 1.0, 'learning_rate': 0.2, 'max_depth': 3, 'min_child_weight': 3, 'n_estimators': 200, 'subsample': 0.6} at: https://dagshub.com/iamprashantjain/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/6/runs/1b67115730f44e1e9fc0eda333cc751f.
2026/04/29 21:09:33 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: https://dagshub.com/iamprashantjain/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/6.
2026/04/29 21:09:42 INFO mlflow.tracking._tracking_service.client: 🏃 View run XGB_{'colsample_bytree': 1.0, 'learning_rate': 0.2, 'max_depth': 3, 'min_child_weight': 3, 'n_estimators': 200, 'subsample': 0.8} at: https://dagshub.com/iamprashantjain/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/6/runs/715cf3409665402eb25287d015f39c0c.
2026/04/29 21:09:42 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: https://dagshub.com/iamprashantjain/Swiggy-Del

   Logged 1251/1296 combinations...


2026/04/29 21:17:51 INFO mlflow.tracking._tracking_service.client: 🏃 View run XGB_{'colsample_bytree': 1.0, 'learning_rate': 0.2, 'max_depth': 7, 'min_child_weight': 1, 'n_estimators': 300, 'subsample': 1.0} at: https://dagshub.com/iamprashantjain/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/6/runs/9bd05a33409d4e02882e821bd2393480.
2026/04/29 21:17:51 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: https://dagshub.com/iamprashantjain/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/6.
2026/04/29 21:18:01 INFO mlflow.tracking._tracking_service.client: 🏃 View run XGB_{'colsample_bytree': 1.0, 'learning_rate': 0.2, 'max_depth': 7, 'min_child_weight': 3, 'n_estimators': 100, 'subsample': 0.6} at: https://dagshub.com/iamprashantjain/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/6/runs/3bc507465fec460ca83b8e0d94c7b079.
2026/04/29 21:18:01 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: https://dagshub.com/iamprashantjain/Swiggy-Del